# MART_P10_TRANSBORDO_REGIOES

**P10. Como as internações transbordam entre as regiões de saúde de Alagoas?** *(grafo)*

Cada linha é uma **aresta** região de residência → região do hospital. `TIPO_FLUXO`:
`MESMA REGIAO`, `MESMA MACRORREGIAO`, `ENTRE MACRORREGIOES` ou `OUTRO ESTADO` (paciente
de fora de Alagoas).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P10_TRANSBORDO_REGIOES
    (REGIAO_ORIGEM, MACRO_ORIGEM, REGIAO_DESTINO, MACRO_DESTINO, TIPO_FLUXO,
     QTD_INTERNACOES, VL_TOTAL)
SELECT mo.REGIAO_SAUDE, mo.MACRORREGIAO, md.REGIAO_SAUDE, md.MACRORREGIAO,
       CASE WHEN mo.UF <> 'AL' THEN 'OUTRO ESTADO'
            WHEN mo.REGIAO_SAUDE = md.REGIAO_SAUDE THEN 'MESMA REGIAO'
            WHEN mo.MACRORREGIAO = md.MACRORREGIAO THEN 'MESMA MACRORREGIAO'
            ELSE 'ENTRE MACRORREGIOES' END,
       SUM(f.QTD_INTERNACAO),
       SUM(f.VL_TOTAL)
FROM FATO_INTERNACAO f
JOIN DIM_MUNICIPIO mo ON mo.SK_MUNICIPIO = f.SK_MUNICIPIO_RESIDENCIA
JOIN DIM_MUNICIPIO md ON md.SK_MUNICIPIO = f.SK_MUNICIPIO_ATENDIMENTO
WHERE f.SK_MUNICIPIO_RESIDENCIA <> -1
  AND md.UF = 'AL'
GROUP BY mo.REGIAO_SAUDE, mo.MACRORREGIAO, md.REGIAO_SAUDE, md.MACRORREGIAO,
         CASE WHEN mo.UF <> 'AL' THEN 'OUTRO ESTADO'
              WHEN mo.REGIAO_SAUDE = md.REGIAO_SAUDE THEN 'MESMA REGIAO'
              WHEN mo.MACRORREGIAO = md.MACRORREGIAO THEN 'MESMA MACRORREGIAO'
              ELSE 'ENTRE MACRORREGIOES' END
"""

In [ ]:
recarregar("MART_P10_TRANSBORDO_REGIOES", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P10_TRANSBORDO_REGIOES FETCH FIRST 10 ROWS ONLY")